# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:
- https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Install the mlcroissant library if not present
!pip install mlcroissant

## 1. Data Loading
Load metadata and explore the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is an object; access properties as attributes

print(f"{metadata.name}: {metadata.description}\nPublished: {metadata.datePublished}")

## 2. Data Overview
List all available **record sets** and their fields using their `@id` values, as required by the Croissant format.

In [ ]:
# List all record sets, fields, and columns by their @id
def display_record_sets_info(dataset):
    print("Available Record Sets and their Fields:")
    for rs in dataset.record_sets:
        print(f"- RecordSet @id: {rs.id} (name: {rs.name})")
        if hasattr(rs, 'fields'):
            for field in rs.fields:
                print(f"   - Field @id: {field.id} (name: {field.name}, dataType: {getattr(field, 'dataType', None)})")
                if hasattr(field, 'columns') and field.columns:
                    for col in field.columns:
                        print(f"     - Column @id: {col.id} (name: {col.name}, dataType: {getattr(col, 'dataType', None)})")
        print()

display_record_sets_info(dataset)

## 3. Data Extraction
Load records from a selected record set into a pandas DataFrame for analysis. All `@id` references are used for clarity and precision.

In [ ]:
"""
For this dataset, we'll extract all available record sets into separate DataFrames.

All RecordSets, their @id, their name, and their fields have been printed in the previous cell.
Below, we collect all record set and field @id values for extraction.
"""
# List the RecordSet @id(s) from the dataset
record_sets_ids = [rs.id for rs in dataset.record_sets]
print("RecordSets found:", record_sets_ids)

# Load all records for each record set into a pandas DataFrame
dataframes = dict()
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet {record_set_id}")
    else:
        print(f"No records found for RecordSet {record_set_id}")

# Show columns in the first available record set
if dataframes:
    chosen_record_set_id = next(iter(dataframes))
    print(f"\nColumns for RecordSet {chosen_record_set_id}:")
    print(dataframes[chosen_record_set_id].columns.tolist())
    display(dataframes[chosen_record_set_id].head(5))
else:
    print("No DataFrames were created -- likely no records available.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing to the main record set. We'll select a numeric field (e.g., *coverage*, *MW*, or *peptide_count*) for demonstration. All columns are referenced by their Croissant `@id`.

In [ ]:
# Replace these values based on the printed overview above, using the relevant @id
record_set_id = chosen_record_set_id  # As determined above
df = dataframes[record_set_id]
print(f"Operating on RecordSet {record_set_id} with columns: {df.columns.tolist()}")

# Identify a numeric field: e.g., look for candidates typical of proteomics datasets
candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
if not candidates:
    # Try parsing columns to numeric to find possible numeric fields (demo for up to 5 fields)
    sample_numeric_cols = []
    for col in df.columns:
        try:
            df[col + "_numeric"] = pd.to_numeric(df[col], errors="coerce")
            if df[col + "_numeric"].notnull().sum() > 0:
                sample_numeric_cols.append(col)
            if len(sample_numeric_cols) >= 5:
                break
        except Exception:
            pass
    if sample_numeric_cols:
        numeric_field_id = sample_numeric_cols[0] + "_numeric"
    else:
        numeric_field_id = None
else:
    numeric_field_id = candidates[0]

if numeric_field_id:
    print(f"Selected numeric field for analysis: {numeric_field_id}")
    series = df[numeric_field_id]
    threshold = np.nanpercentile(series, 80)  # upper 20% for demo

    filtered_df = df[series > threshold].copy()
    print(f"Records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Choose a group field (non-numeric and not ID)
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and col.endswith('id') is False and df[col].dtype == object and len(df[col].unique()) < 20:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
        display(grouped_df.head())
    else:
        print("No suitable group field found for demonstration.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize distributions or relationships. For demonstration, we'll plot a histogram of the selected numeric field and a bar chart for the group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id], kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        order = df[group_field].value_counts().index[:10]
        sns.boxplot(x=group_field, y=numeric_field_id, data=df, order=order)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.tight_layout()
        plt.show()


## 6. Conclusion
In this notebook, we've used the `mlcroissant` library to discover, load, and explore the FAIR² mass spectrometry dataset, referencing all Croissant entities by their `@id`. We:
- Listed available RecordSets and fields by `@id`
- Loaded tabular records into pandas DataFrames
- Processed and visualized a numeric field
This workflow can be extended for more advanced analyses and used as a template for other Croissant datasets.